# Runpod Environment Setup

SSH implements public-key cryptography to establish trust between client and server systems. In this paradigm, we have the **public key** (`.pub`) that functions as an access *verifier*, installed on target systems (like Runpod nodes), and the **private key** serves as the unique authentication factor for mathematically proving identity.

Runpod automatically loads the public keys to the pods which makes SSH access possible (see @fig-runpod-ssh). Next, we want read/write access to remote repositories in GitHub. For security reasons, we use GitHub [personal access token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens) (PAT) which we store securely in Runpod **secrets manager**. Another common use-case of secrets are for storing API keys for LLM inference providers which can later be loaded into pods as **environmental variables**.

![**Setting up pods for development.** Mainly involves giving (1) *us* access to pods, and (2) *pods* access to external services (e.g. GitHub, OpenAI, etc). Not shown here is configuring the development environment (virtual envs, installing required libraries) once we have the initial code + further tooling (appendices).](./img/runpod-ssh.png){#fig-runpod-ssh}

:::{.callout-warning}
## Docker development limitation
Runpod is architecturally incompatible with standard Docker workflows (i.e. pods are themselves containers and cannot host a Docker daemon). For any development requiring local container orchestration or image building, use a traditional cloud VM (e.g. AWS EC2, GCP Compute Engine) where you have full control over the host machine.
:::

## Give local SSH access to pods

Generate SSH keys for Runpod (no passphrase):
```{.bash filename="$ (local)"}
ssh-keygen -t ed25519 -C "runpod" -f ~/.ssh/id_ed25519_runpod -N ""
cat ~/.ssh/id_ed25519_runpod.pub
```
Append the output to **Settings**> **SSH Public Keys** in Runpod (separated by newline).

## Give pods SSH access to GitHub

Go to **Settings>Developer Settings** in GitHub to generate a fine-grained **personal access token** (PAT).
Make sure to apply the appropriate repository permissions. It is also good practice to only allow access to select 
repositories: 

![](./img/runpod-github-pat.png)

Add the generated token to Runpod **secrets manager**:

![](./img/runpod-secrets-manager.png)


## Pod creation

During pod creation, load secrets (e.g. the GitHub PAT) as **environmental variable**. Note that we reduce the container disk
to the minimum required value since we typically work on the **pod volume** mounted on `/workspace` which persists data 
even when the pod is stopped[^volumecost]. Also select the appropriate image (here we choose the image with PyTorch 2.8 support):

![](./img/runpod-pod-creation.png)

This is also the place to load other API keys.
Once the env vars are added,
connect using **SSH over exposed TCP** [connection string]{.underline}[^connstring]. Inside the pod you can check
that the access tokens have been loaded using:
```{.bash filename="$ (pod)"}
env | grep GITHUB
```

Then, we can clone [our repository](https://github.com/particle1331/ai-notebooks) via HTTPS with the required access using:
```{.bash filename="$ (pod)"}
cd /workspace
export USERNAME=particle1331
export GITHUB_REPO=github.com/particle1331/ai-notebooks
git clone https://${USERNAME}:${GITHUB_TOKEN}@${GITHUB_REPO}.git
cd ai-notebooks
```

:::{.callout-tip}
You can try `git push` to test for write access as this would fail if the PAT was improperly loaded.
:::

[^connstring]: e.g. `ssh root@63.141.33.33 -p 22011 -i ~/.ssh/id_ed25519_runpod`

[^volumecost]: A stopped pod still has an hourly cost (e.g. 0.01$ / hr) so it's a good idea to terminate a pod before a long break. Note that this means losing the data in `/workspace`.

## Virtual environment

Our recommended approach is to use `uv` to build a venv at `.venv` that is synced using `uv.lock`. This environment can then be used as Jupyter kernel [in vscode](https://code.visualstudio.com/docs/datascience/jupyter-kernel-management). Or run scripts using [`uv run`](https://docs.astral.sh/uv/concepts/projects/run/#running-scripts). The required Python version is also specified ([-@lst-Makefile]):

```{.bash filename="$ (pod)"}
make venv
```

:::{.callout-tip}
Some environments can cause `uv` failures (e.g. servers with network-mounted filesystems like AzureML compute instances). 
In this situation, you can use `pip` to install packages without using `uv` as dependency manager:
```bash
# install required python version, pip, and activate venv
make uv
uv python install 3.13
uv venv .venv && source .venv/bin/activate
curl -sS https://bootstrap.pypa.io/get-pip.py | .venv/bin/python

# install requirements on venv
make requirements
uv pip install -r requirements.txt
uv pip install -e .
```
:::

:::{.callout-note}
In the case of short-lived pods, you don't necessarily need to perfectly setup an environment or maintain a clean state. However, reproducibility benefits from a controlled, well-defined environment. As usual, the level of precision needed depends on the scope of the project.
:::

## Appendix: Quarto docs

Check [here](https://github.com/particle1331/ai-notebooks/blob/main/.github/workflows/publish.yml) for the Quatro version that we're using. Adjust the following variable accordingly:
```{.bash filename="$ (pod)"}
export QUARTO_VERSION=1.7.32  # may be outdated
wget -q https://github.com/quarto-dev/quarto-cli/releases/download/v${QUARTO_VERSION}/quarto-${QUARTO_VERSION}-linux-amd64.deb
dpkg -i quarto-${QUARTO_VERSION}-linux-amd64.deb && rm quarto-${QUARTO_VERSION}-linux-amd64.deb
```

Preview then [port-forward](https://code.visualstudio.com/docs/remote/ssh#_forwarding-a-port-creating-ssh-tunnel) using vscode:
```{.bash filename="$ (pod)"}
make docs
```

## Appendix: Tmux

Tmux allows you to open multiple windows in a **single SSH session** to a remote server, without needing to authenticate separately for each window. You can also detach from a tmux session and reconnect later to resume exactly where you left off. This means you can run several tasks in parallel, switch between them easily, and keep them running even if your connection drops. ◧

For convenience, I use [byobu](https://www.byobu.org/). First, [install byobu](https://www.byobu.org/downloads) in the remote server. This will run a **tmux daemon** on the server which persists the session. In the following table, I list commands and workflows I found useful:

```{.bash filename="$ (pod)"}
apt-get update
apt-get install -y byobu
byobu
```

| Command | Function |
| :--: | :-- |
| F1, Shift + F1 | Display help |
| Ctrl + F2 | Create new split vertically |
| Shift + F2 | Create new split horizontally |
| F2 | Create new window |
| F8 | Rename the current window |
| Ctrl + F6 | Kill a focused split |
| F3 / F4 | Switch between windows |
| Shift + F3 / F4 | Switch between splits |
| F6 | Detach session |
| Shift + F9 | Run command on all splits. See @fig-byobu-command-splits. |
: Byobu commands {tbl-colwidths="[30,70]"}

:::{.callout-caution}
Make sure to disable your system keyboard shortcuts for these combinations! For example, (ꐦ¬_¬) Apple has a default shortcut for ⌃F2, which translates to Ctrl+F2, resulting in some head-scratching if you're unaware! (It similarly has shortcuts for ⌃F3, ⌃F6, etc.) 
:::

Running the same command on multiple splits (very useful):

![Running a dynamic command on 3 splits.](./img/byobu-commands.png){#fig-byobu-command-splits}

**Resuming.** Working on a remote server, you might lose your connection unexpectedly or intentionally disconnect while wanting to keep processes running. In such cases, you can **press F6** in Byobu to detach the session and logout. Once you SSH again to the server, the session will be restored by running `byobu`. This is very useful!

![Resuming a detached session. The process kept running in the background while we were away.](./img/byobu-detached.png){#fig-byobu-command-splits}

## Appendix: Code listings

::: {#lst-Makefile lst-cap="Makefile for the project."}
```{.bash filename=Makefile}
{{< include "../../Makefile" >}}
```
:::